# Prepare TRENDYv10 inputs for Figures 1–4

Adapted from `TRENDY_extraction_flux_locations_sept14_codex.ipynb` supplied by Ngoc Nguyen.
Run cells from top to bottom in the `source-sink-dynamics` Python environment.
First edit **Configuration** below. No data are downloaded automatically.

1. Obtain **TRENDYv10 / Global Carbon Budget 2021** gridded model output through the
   [Global Carbon Budget Data Hub](https://globalcarbonbudget.org/datahub/) and its linked
   [model-data browser](https://mdosullivan.github.io/GCB/). Follow the provider's access instructions
   if your requested model or experiment is not directly downloadable. Use v10, not the latest release.
2. Download and unpack the selected models, scenarios and variables into `DATA_DIR` as
   `<MODEL>/<MODEL>_<SCENARIO>_<VARIABLE>.nc`, or edit `INPUT_TEMPLATE` to match your layout.
   Regional aggregates and summary spreadsheets are not substitutes for these gridded files.
3. Supply the Cabon site table at `SITE_FILE`, then run this notebook.
4. Run the four R scripts. Default `OUTPUT_DIR` matches their input directory.
   If overriding it, place/copy the outputs under the R root's
   `results/TRENDYv10/31_site_weighted/` before running the figures.

**Method preserved:** use on-site ring-width sites, choose their nearest rectilinear grid cells,
count each occupied cell once, calculate an equal-weight mean of available time steps per calendar year,
and multiply by grid-cell area. Units are **source units × m²**, not an area-normalized average or
an annual flux integral. No unit conversion or duration weighting is performed. Missing and zero
values at selected cells are retained for the figure scripts to handle.

Coordinates are normalized to ascending latitude and longitude in [−180, 180).
Latitude is never wrapped or sign-flipped. Areas use valid supplied latitude bounds or midpoint
estimates clipped to the poles. Gaussian-grid areas without valid bounds are approximate.
Only rectilinear grids with globally regular longitude spacing are supported. Integer month offsets
use their origin month; other timestamps use their declared calendar. Time bounds are not used.

The saved audit describes **this run's actual files**. Existing matching outputs are overwritten;
a failed run may leave outputs from completed jobs, so require a successful final cell and check the
manifest before using results. For a trial run, choose a separate `OUTPUT_DIR`.

The original notebook's numerical extraction functions are retained; configuration, validation,
site mappings and run metadata have been added. This repository copy deliberately has no saved
execution output. See the repository README for setup and the full workflow.


## 1. Imports
Install dependencies using `conda env create -f environment.yml` from the repository root.


In [ ]:
from pathlib import Path
import os
import multiprocessing
import re
import numpy as np
import pandas as pd
import xarray as xr
from joblib import Parallel, delayed, parallel_backend


## 2. Configuration
Change paths and selections here. Environment-variable overrides are optional. `N_JOBS = 1` runs sequentially; the default is at most two workers.


In [ ]:
# Edit this cell before running the notebook. All paths can be absolute.
# PROJECT must be the repository root containing code/ and data/.
# Auto-detection supports Jupyter launched from the root or code/ directory.
project_default = Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd()
PROJECT = Path(os.environ.get('SOURCE_SINK_ROOT', str(project_default))).expanduser().resolve()

# Download and unpack TRENDYv10 NetCDF files into DATA_DIR (see README).
DATA_DIR = Path(os.environ.get('TRENDY_DATA_DIR',
    str(PROJECT / 'data' / 'TRENDYv10' / 'downloads'))).expanduser().resolve()
SITE_FILE = Path(os.environ.get('TRENDY_SITE_FILE',
    str(PROJECT / 'data' / 'cabon' / 'Cabonetal_site_info.csv'))).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get('TRENDY_OUTPUT_DIR',
    str(PROJECT / 'results' / 'TRENDYv10' / '31_site_weighted'))).expanduser().resolve()

# Defaults cover the models selected by the four figure scripts.
# Add other downloaded models, e.g. CLM5.0, ORCHIDEEv3, CLASSIC, CLASSIC-N, LPJ-GUESS.
MODELS = ['ISBA-CTRIP', 'LPX-Bern', 'CABLE-POP', 'ORCHIDEE']
SCENARIOS = ['S0', 'S1', 'S2']
VARIABLES = ['cLeaf', 'cRoot', 'cWood', 'gpp', 'ra']
START_YEAR = None  # Inclusive lower limit; None preserves all available years.
N_JOBS = min(2, max(1, multiprocessing.cpu_count() - 1))  # 1 = sequential.

# Expected input: DATA_DIR/<MODEL>/<MODEL>_<SCENARIO>_<VARIABLE>.nc
# Modify only this template if your downloaded files have a different layout.
INPUT_TEMPLATE = '{model}/{model}_{scenario}_{variable}.nc'

for name, value in [('PROJECT', PROJECT), ('DATA_DIR', DATA_DIR),
                    ('SITE_FILE', SITE_FILE), ('OUTPUT_DIR', OUTPUT_DIR)]:
    print(f'{name}: {value}')


## 3. Extraction helpers


In [ ]:
COORDINATE_NAMES = {
    'lat': ('lat', 'latitude', 'Latitude', 'LATITUDE', 'lat_FULL', 'Y', 'y'),
    'lon': ('lon', 'longitude', 'Longitude', 'LONGITUDE', 'lon_FULL', 'X', 'x'),
    'time': ('time', 'time_counter', 'Time', 'TIME'),
}


def normalize_coordinates(ds):
    """Standardize a rectilinear degree grid; keep data attached while sorting."""
    rename = {}
    for target, candidates in COORDINATE_NAMES.items():
        source = next((name for name in candidates if name in ds.variables), None)
        if source is None:
            raise ValueError(f'Missing {target} coordinate')
        if source != target:
            rename[source] = target
    ds = ds.rename(rename)
    for axis in ('lat', 'lon'):
        coord = ds[axis]
        if coord.ndim != 1:
            raise ValueError(f'{axis}: only 1-D rectilinear grids are supported')
        units = str(coord.attrs.get('units', '')).lower()
        if 'degree' not in units:
            raise ValueError(f'{axis}: expected degree units, found {units!r}')
        if coord.dims != (axis,):
            ds = ds.swap_dims({coord.dims[0]: axis})
        values = ds[axis].values
        if values.size < 2 or not np.isfinite(values).all():
            raise ValueError(f'{axis}: coordinates must contain finite grid centers')
    if np.any(np.abs(ds.lat.values) > 90):
        raise ValueError('Latitude outside [-90, 90]; do not wrap latitude')
    lon = ds.lon.values
    if not (np.all((lon >= -180) & (lon <= 180)) or
            np.all((lon >= 0) & (lon <= 360))):
        raise ValueError('Unrecognized longitude range; inspect this grid')
    if np.any(lon >= 180):
        attrs = dict(ds.lon.attrs)
        ds = ds.assign_coords(lon=((ds.lon + 180) % 360) - 180)
        ds.lon.attrs = attrs
    ds = ds.sortby(['lat', 'lon'])
    for axis in ('lat', 'lon'):
        if np.any(np.diff(ds[axis].values) <= 0):
            raise ValueError(f'{axis}: duplicate coordinates after normalization')
    return ds


def year_labels(time):
    """Extract years without pretending that all monthly files use 360-day years."""
    units = str(time.attrs.get('units', '')).strip()
    calendar = str(time.attrs.get('calendar', 'standard')).lower()
    values = np.asarray(time.values)
    match = re.fullmatch(r'months since\s+(\d+)-\d+-\d+(?:[ T].*)?', units)
    if match:
        if not np.isfinite(values).all() or not np.allclose(values, np.rint(values)):
            raise ValueError('Fractional months require an explicit calendar interpretation')
        # The input month matters when an origin is not January.
        origin_month = int(units.split('since', 1)[1].strip().split('-')[1])
        return int(match[1]) + (origin_month - 1 + np.rint(values).astype(int)) // 12
    if calendar in ('noleap', '365_day') and units.startswith('years since'):
        units = units.replace('years since', 'common_years since', 1)
    from netCDF4 import num2date
    return np.array([d.year for d in num2date(values, units=units, calendar=calendar)])


def cell_areas(ds):
    """Spherical areas for this notebook's global, regularly spaced longitude grids."""
    lat = ds.lat.values.astype(float)
    lon = ds.lon.values.astype(float)
    spacing = np.diff(lon)
    dlon = float(np.median(spacing))
    if not np.allclose(spacing, dlon, rtol=1e-4, atol=1e-5):
        raise ValueError('Irregular longitude spacing needs explicit cell bounds')
    if not np.isclose(dlon * len(lon), 360, atol=1e-3):
        raise ValueError('Expected a global longitude grid')
    # Validate supplied bounds before using them: some ISBA-CTRIP files
    # contain zero-width bounds unrelated to their latitude centers.
    bounds_name = ds.lat.attrs.get('bounds')
    valid_bounds = False
    if bounds_name and bounds_name in ds:
        bounds = ds[bounds_name].transpose('lat', ...).values
        if bounds.shape == (len(lat), 2) and np.isfinite(bounds).all():
            lower, upper = np.min(bounds, axis=1), np.max(bounds, axis=1)
            valid_bounds = bool(
                np.all(upper > lower)
                and np.all(lower >= -90) and np.all(upper <= 90)
                and np.all(lower <= lat) and np.all(lat <= upper))
        if not valid_bounds:
            if not np.allclose(np.diff(lat), np.median(np.diff(lat)),
                               rtol=1e-4, atol=1e-5):
                raise ValueError('Invalid latitude bounds on an irregular grid; '
                                 'inspect the source before estimating areas')
            import warnings
            warnings.warn(
                f'{bounds_name}: invalid latitude bounds; deriving bounds '
                'from the regular latitude centers.', RuntimeWarning)
    if not valid_bounds:
        edges = np.r_[lat[0] - (lat[1] - lat[0]) / 2,
                      (lat[:-1] + lat[1:]) / 2,
                      lat[-1] + (lat[-1] - lat[-2]) / 2]
        lower, upper = edges[:-1], edges[1:]
    lower, upper = np.clip(lower, -90, 90), np.clip(upper, -90, 90)
    area = 6371000.0**2 * np.deg2rad(dlon) * (
        np.sin(np.deg2rad(upper)) - np.sin(np.deg2rad(lower)))
    if not np.isfinite(area).all() or np.any(area <= 0):
        raise ValueError('Cell areas must be finite and strictly positive')
    return xr.DataArray(area, dims='lat', coords={'lat': ds.lat})


def site_cell_indices(ds, sites):
    """Return latitude/longitude indices for each site in its input order."""
    lat_points = sites['Lat.'].to_numpy(dtype=float)
    lon_points = sites['Lon.'].to_numpy(dtype=float)
    if not np.isfinite(lat_points).all() or not np.isfinite(lon_points).all():
        raise ValueError('Missing or non-finite site coordinates')
    if np.any(np.abs(lat_points) > 90) or np.any(np.abs(lon_points) > 180):
        raise ValueError('Expected site latitude [-90, 90] and longitude [-180, 180]')
    # Check latitude coverage using outer cell edges, including partial grids.
    lat = ds.lat.values
    lower = max(-90, lat[0] - (lat[1] - lat[0]) / 2)
    upper = min(90, lat[-1] + (lat[-1] - lat[-2]) / 2)
    if np.any((lat_points < lower) | (lat_points > upper)):
        raise ValueError('A site is outside the model latitude coverage')
    iy = np.abs(lat[:, None] - lat_points).argmin(axis=0)
    angular_distance = np.abs((ds.lon.values[:, None] - lon_points + 180) % 360 - 180)
    ix = angular_distance.argmin(axis=0)
    return iy, ix


def site_mapping(ds, sites):
    """Keep site IDs even when several sites occupy the same model cell."""
    iy, ix = site_cell_indices(ds, sites)
    return pd.DataFrame({'lat': ds.lat.values[iy], 'lon': ds.lon.values[ix],
                         'SITE_ID': sites['Site'].to_numpy()})


def nearest_cells(ds, sites):
    """Select each occupied cell once, preserving the original extraction rule."""
    iy, ix = site_cell_indices(ds, sites)
    pairs = np.unique(np.column_stack([iy, ix]), axis=0)
    return {axis: xr.DataArray(pairs[:, j], dims='cell')
            for j, axis in enumerate(('lat', 'lon'))}


def extract_yearly_cells(ds, variable, sites, start_year=None):
    ds = normalize_coordinates(ds)
    if set(ds[variable].dims) != {'time', 'lat', 'lon'}:
        raise ValueError(f'{variable}: expected only time, lat, lon dimensions')
    years = year_labels(ds.time)
    indexers = nearest_cells(ds, sites)
    # Subset before aggregation: avoid a time x global-grid mask and empty CSV rows.
    selected = ds[variable].isel(indexers).assign_coords(year=('time', years))
    if start_year is not None:
        selected = selected.isel(time=np.flatnonzero(years >= start_year))
    if selected.sizes['time'] == 0:
        raise ValueError('No time steps in the requested period')
    # Read only the selected sites once. Grouping lazy NetCDF data can
    # repeatedly read/decompress source chunks for individual years.
    selected = selected.load()
    annual = selected.groupby('year').mean('time', skipna=True)
    weighted = annual * cell_areas(ds).isel(lat=indexers['lat'])
    frame = weighted.rename(variable).to_dataframe().reset_index()
    return frame[['year', 'lat', 'lon', variable]]


## 4. Validate inputs and audit grids
This checks all selected files and builds the Figure 1 site mappings before any results are written.


In [ ]:
# Validate all requested files before writing any extraction output.
if not (PROJECT / 'code').is_dir():
    raise ValueError('Set PROJECT to the repository root containing code/.')
if not isinstance(N_JOBS, int) or isinstance(N_JOBS, bool) or N_JOBS < 1:
    raise ValueError('N_JOBS must be a positive integer.')
if START_YEAR is not None and (not isinstance(START_YEAR, int) or isinstance(START_YEAR, bool)):
    raise ValueError('START_YEAR must be an integer year or None.')
for name, values in [('MODELS', MODELS), ('SCENARIOS', SCENARIOS), ('VARIABLES', VARIABLES)]:
    if not values or len(values) != len(set(values)):
        raise ValueError(f'{name} must be nonempty and contain no duplicates.')
if not SITE_FILE.is_file():
    raise FileNotFoundError(f'Missing SITE_FILE: {SITE_FILE}')
sites = pd.read_csv(SITE_FILE)
required = {'Site', 'Lat.', 'Lon.', 'On-site RW'}
if not required.issubset(sites.columns):
    raise ValueError(f'SITE_FILE is missing columns: {sorted(required - set(sites.columns))}')
sites = sites.loc[sites['On-site RW'].eq(True)].copy()
if sites.empty or sites['Site'].isna().any() or sites['Site'].duplicated().any():
    raise ValueError('Expected nonempty on-site observations with unique, nonmissing Site IDs.')

jobs = [(model, scenario, variable,
         DATA_DIR / INPUT_TEMPLATE.format(model=model, scenario=scenario, variable=variable))
        for model in MODELS for scenario in SCENARIOS for variable in VARIABLES]
missing = [str(path) for _, _, _, path in jobs if not path.is_file()]
if missing:
    raise FileNotFoundError('Download/unpack the requested TRENDY files into DATA_DIR. Missing:\n'
                            + '\n'.join(missing))

# Audit the actual inputs; no audit result from the author's machine is assumed.
coordinate_rows = []
model_grids = {}
site_mappings = {}
for model, scenario, variable, path in jobs:
    with xr.open_dataset(path, decode_times=False) as raw:
        checked = normalize_coordinates(raw)
        if variable not in checked or set(checked[variable].dims) != {'time', 'lat', 'lon'}:
            raise ValueError(f'{path}: expected {variable} with time/lat/lon dimensions.')
        cell_areas(checked)  # Validate area assumptions before extraction.
        year_labels(checked.time)
        grid = (checked.lat.values.copy(), checked.lon.values.copy())
        if model in model_grids:
            if not all(np.array_equal(a, b) for a, b in zip(grid, model_grids[model])):
                raise ValueError(f'{path}: grid differs across this model; figure joins require matching coordinates.')
        else:
            model_grids[model] = grid
            site_mappings[model] = site_mapping(checked, sites)
        row = dict(model=model, scenario=scenario, variable=variable,
                   source_file=str(path), source_units=str(raw[variable].attrs.get('units', 'unspecified')),
                   time_units=str(checked.time.attrs.get('units', '')),
                   calendar=str(checked.time.attrs.get('calendar', 'standard')))
        for axis in ('lat', 'lon'):
            row[f'{axis}_min'] = float(checked[axis].min())
            row[f'{axis}_max'] = float(checked[axis].max())
            row[f'{axis}_count'] = int(checked[axis].size)
        coordinate_rows.append(row)
coordinate_audit = pd.DataFrame(coordinate_rows)
display(coordinate_audit)
print(f'Validated {len(jobs)} files for {len(sites)} observed sites.')


## 5. Extract and save
One job per file. The final cell writes site mappings, the coordinate audit and a run manifest with package versions and output row counts.


In [ ]:
def process_and_save_variable(model, scenario, variable, input_path, output_root,
                              sites, start_year):
    """Read, extract, and save one file within an isolated worker process."""
    output_root = Path(output_root)
    output_dir = output_root / model
    output_dir.mkdir(parents=True, exist_ok=True)
    path = Path(input_path)
    # Each worker opens and closes its own dataset; no file handles are shared.
    import time
    started = time.monotonic()
    print(f'Starting {model} {scenario} {variable}', flush=True)
    try:
        with xr.open_dataset(path, decode_times=False) as ds:
            frame = extract_yearly_cells(ds, variable, sites, start_year)
    except Exception as exc:
        raise RuntimeError(f'Extraction failed for {path}') from exc
    output_file = output_dir / f'{variable}_{scenario}_31_site_weighted_yearly_mean.csv'
    temporary_file = output_file.with_suffix('.csv.tmp')
    frame.to_csv(temporary_file, index=False)
    os.replace(temporary_file, output_file)
    message = (f'Saved {model} {scenario} {variable}: {len(frame):,} rows '
               f'in {time.monotonic() - started:.1f}s')
    print(message, flush=True)
    return dict(model=model, scenario=scenario, variable=variable,
                input=str(path), output=str(output_file), rows=len(frame),
                first_year=int(frame.year.min()), last_year=int(frame.year.max()),
                input_bytes=path.stat().st_size, input_mtime_ns=path.stat().st_mtime_ns)


In [ ]:
# Each worker opens its own NetCDF file. Existing matching CSVs are replaced.
with parallel_backend('loky', inner_max_num_threads=1):
    saved_files = Parallel(n_jobs=N_JOBS, batch_size=1,
                           pre_dispatch=N_JOBS, verbose=50)(
        delayed(process_and_save_variable)(
            model, scenario, variable, path, OUTPUT_DIR, sites, START_YEAR)
        for model, scenario, variable, path in jobs
    )

# These top-level mappings are required by Figure1_code.R.
for model, mapping in site_mappings.items():
    mapping.to_csv(OUTPUT_DIR / f'{model}_site_info.csv', index=False)
coordinate_audit.to_csv(OUTPUT_DIR / 'coordinate_audit.csv', index=False)

import json
import platform
from datetime import datetime, timezone
from importlib.metadata import version
manifest = {
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'project': str(PROJECT), 'data_dir': str(DATA_DIR), 'site_file': str(SITE_FILE),
    'output_dir': str(OUTPUT_DIR), 'models': MODELS, 'scenarios': SCENARIOS,
    'variables': VARIABLES, 'start_year': START_YEAR, 'n_jobs': N_JOBS,
    'input_template': INPUT_TEMPLATE, 'python': platform.python_version(),
    'packages': {name: version(name) for name in ['numpy', 'pandas', 'xarray', 'netCDF4', 'joblib']},
    'method': 'Equal-weight annual mean of available time steps multiplied by cell area (m2).',
    'files': saved_files,
}
(OUTPUT_DIR / 'extraction_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
display(pd.DataFrame(saved_files))
print(f'Finished. Outputs are in {OUTPUT_DIR}')
